<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Opoznienia_KMK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza Opóźnień Komunikacji Miejskiej w Krakowie (GTFS-RT)

Niniejszy notatnik służy do analizy rzeczywistych opóźnień komunikacji miejskiej w Krakowie. Dane pobierane są w czasie rzeczywistym z usług [GTFS-RT ZTP Kraków](https://gtfs.ztp.krakow.pl/).

Wykorzystujemy:
- **TripUpdates** (format `.pb` - Protobuf), aby pozyskać estymowane czasy przyjazdów i porównać je do planowanych.
- **Dane statyczne (GTFS)** do podpięcia lokalizacji geo (przystanków) oraz nazw linii.

Notatnik wygeneruje interaktywne mapy ulic ukazujące natężenie opóźnień, a także odpowiednie statystyki i wykresy.


Dane pokazują tylko aktualne zmiany, a nie trend historyczny. Opóźnienia z systemu mogą różnić się od rzeczywistych.

Wyniki programu będą się znacząco róźnić w zależności od pory dnia.

In [11]:
# Instalacja/importy – działa w Colabie, Jupyterze i lokalnym venv bez składni specyficznej dla notebooka.
import sys
import subprocess
import importlib
import warnings
import datetime
import os
import re
import time
import io
import zipfile
import urllib.request


def ensure_import(import_name, package_name=None):
    """Importuje pakiet, a gdy go brakuje – doinstalowuje go przez pip."""
    package_name = package_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f"Instaluję brakujący pakiet: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return importlib.import_module(import_name)


# Pakiety lekkie/importowane bezpośrednio
ensure_import("requests")
ensure_import("pandas")
ensure_import("plotly")
ensure_import("folium")
ensure_import("matplotlib")
ensure_import("networkx")
ensure_import("osmnx")
ensure_import("scipy")
ensure_import("google.transit.gtfs_realtime_pb2", "gtfs-realtime-bindings")

import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import TimestampedGeoJson
import osmnx as ox
import networkx as nx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

LOCAL_TZ = "Europe/Warsaw"
pd.set_option("display.max_columns", 100)

print("Środowisko gotowe ✅")


Środowisko gotowe ✅


In [12]:
def _safe_has_field(proto_message, field_name):
    """Bezpieczny HasField – w niektórych wersjach protobuf skalar nie ma presence."""
    try:
        return proto_message.HasField(field_name)
    except Exception:
        return bool(getattr(proto_message, field_name, None))


def fetch_gtfs_rt_delays(only_late=False):
    """Pobiera aktualne TripUpdates GTFS-RT i zwraca tabelę obserwacji opóźnień.

    only_late=False zostawia także odjazdy/przyjazdy punktualne lub przed czasem.
    Do wykresów niżej i tak preferujemy dodatnie opóźnienia, ale dzięki temu notebook
    nie robi pustych wykresów, gdy w danym momencie nie ma spóźnionych pojazdów.
    """
    print("Pobieranie aktualnych danych GTFS-RT (TripUpdates) dla Krakowa...")

    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/TripUpdates_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/TripUpdates_A.pb",
    }

    rows = []

    for v_type, url in urls.items():
        print(f"  • Pobieranie danych dla: {v_type}...")
        max_retries = 3

        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
                response.raise_for_status()
                content = response.content

                if not content:
                    raise ValueError("Odpowiedź API jest pusta.")
                if content[:200].lstrip().startswith(b"<"):
                    raise ValueError("Otrzymano HTML zamiast protobuf – serwer ZTP chwilowo zwrócił stronę błędu.")

                feed = gtfs_realtime_pb2.FeedMessage()
                feed.ParseFromString(content)

                for entity in feed.entity:
                    if not entity.HasField("trip_update"):
                        continue

                    trip_update = entity.trip_update
                    trip = trip_update.trip
                    trip_id = str(getattr(trip, "trip_id", "") or "")
                    route_id = str(getattr(trip, "route_id", "") or "")
                    vehicle_id = ""
                    if trip_update.HasField("vehicle"):
                        vehicle_id = str(getattr(trip_update.vehicle, "id", "") or "")

                    for stu in trip_update.stop_time_update:
                        stop_id = str(getattr(stu, "stop_id", "") or "").strip()
                        if not stop_id:
                            continue

                        delay = None
                        event_time = None

                        if stu.HasField("arrival"):
                            if _safe_has_field(stu.arrival, "delay"):
                                delay = stu.arrival.delay
                            if _safe_has_field(stu.arrival, "time"):
                                event_time = stu.arrival.time

                        if delay is None and stu.HasField("departure"):
                            if _safe_has_field(stu.departure, "delay"):
                                delay = stu.departure.delay
                            if event_time is None and _safe_has_field(stu.departure, "time"):
                                event_time = stu.departure.time

                        if delay is None:
                            continue

                        rows.append(
                            {
                                "typ": v_type,
                                "trip_id": trip_id,
                                "vehicle_id": vehicle_id,
                                "line_num": str(route_id or trip_id).strip(),
                                "stop_sequence": int(getattr(stu, "stop_sequence", 0) or 0),
                                "stop_id": stop_id,
                                "delay_sec": float(delay),
                                "delay_min": float(delay) / 60.0,
                                "time": int(event_time) if event_time else None,
                            }
                        )

                break

            except Exception as exc:
                print(f"    Błąd {v_type}, próba {attempt}/{max_retries}: {exc}")
                if attempt < max_retries:
                    time.sleep(2)

    df = pd.DataFrame(rows)

    if df.empty:
        print("Nie pobrano żadnych obserwacji z TripUpdates. Spróbuj ponownie za chwilę.")
        return df

    df["delay_sec"] = pd.to_numeric(df["delay_sec"], errors="coerce")
    df["delay_min"] = df["delay_sec"] / 60.0
    # Odfiltrujmy błędy danych ZTP (np. opóźnienia rzędu 300 minut)
    df = df[(df["delay_min"] >= -10) & (df["delay_min"] <= 120)]
    df = df.dropna(subset=["delay_sec"])

    # Timestamp z GTFS-RT jest epoką Unix – konwertujemy do czasu warszawskiego.
    dt = pd.to_datetime(df["time"], unit="s", utc=True, errors="coerce")
    df["datetime"] = dt.dt.tz_convert(LOCAL_TZ)
    now_waw = pd.Timestamp.now(tz=LOCAL_TZ)
    df["datetime"] = df["datetime"].fillna(now_waw)
    df["hour"] = df["datetime"].dt.strftime("%Y-%m-%dT%H:00:00")
    df["is_late"] = df["delay_sec"] > 0

    if only_late:
        df = df[df["is_late"]].copy()

    late_count = int((df["delay_sec"] > 0).sum())
    print(f"\nZakończono. Pobrano {len(df)} obserwacji, w tym {late_count} z dodatnim opóźnieniem.")
    return df.reset_index(drop=True)


# Dane surowe: zachowujemy wszystko, żeby późniejsze komórki nie robiły pustych wykresów bez powodu.
df_delays_raw = fetch_gtfs_rt_delays(only_late=False)

# Do analizy preferujemy dodatnie opóźnienia. Jeśli akurat nie ma żadnych, zostawiamy wszystkie obserwacje diagnostycznie.
if not df_delays_raw.empty:
    df_delays = df_delays_raw[df_delays_raw["delay_sec"] > 0].copy()
    if df_delays.empty:
        print("Brak dodatnich opóźnień – do wykresów używam wszystkich obserwacji z aktualnego snapshotu.")
        df_delays = df_delays_raw.copy()
    display(df_delays.head())
else:
    df_delays = pd.DataFrame()


Pobieranie aktualnych danych GTFS-RT (TripUpdates) dla Krakowa...
  • Pobieranie danych dla: Tramwaje...
  • Pobieranie danych dla: Autobusy...

Zakończono. Pobrano 617 obserwacji, w tym 298 z dodatnim opóźnieniem.


,typ,trip_id,vehicle_id,line_num,stop_sequence,stop_id,delay_sec,delay_min,time,datetime,hour,is_late
0,Autobusy,20260520_5_123906998_10,PA105,176,2,17598,32.0,0.533333,1780032632,2026-05-29 07:30:32+02:00,2026-05-29T07:00:00,True
1,Autobusy,20260520_5_123906998_10,PA105,176,3,13316,32.0,0.533333,1780032692,2026-05-29 07:31:32+02:00,2026-05-29T07:00:00,True
2,Autobusy,20260520_5_123906998_10,PA105,176,4,10178,32.0,0.533333,1780032872,2026-05-29 07:34:32+02:00,2026-05-29T07:00:00,True
3,Autobusy,20260520_5_123906998_10,PA105,176,5,10180,32.0,0.533333,1780032932,2026-05-29 07:35:32+02:00,2026-05-29T07:00:00,True
4,Autobusy,20260520_5_123906998_10,PA105,176,6,10182,32.0,0.533333,1780033052,2026-05-29 07:37:32+02:00,2026-05-29T07:00:00,True


In [13]:
def normalize_id(value):
    """Normalizuje identyfikatory GTFS/GTFS-RT do porównywalnego tekstu."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return None
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


def id_candidates(value):
    """Zwraca kilka wariantów klucza, bo w Krakowie RT potrafi odnosić się do stop_code,
    a statyczny GTFS ma też stop_id / parent_station / czasem warianty z separatorami.
    """
    s = normalize_id(value)
    if s is None:
        return []

    candidates = [(s, 0, "exact")]
    for sep in ["_", ":", "/", "-", "."]:
        if sep in s:
            parts = [p for p in s.split(sep) if p]
            if parts:
                candidates.append((parts[0], 1, f"before_{sep}"))
                candidates.append((parts[-1], 2, f"after_{sep}"))

    nums = re.findall(r"\d+", s)
    for n in nums:
        candidates.append((n, 3, "numeric"))

    # deduplikacja z zachowaniem najlepszego rankingu
    best = {}
    for key, rank, src in candidates:
        key = normalize_id(key)
        if key and (key not in best or rank < best[key][0]):
            best[key] = (rank, src)
    return [(key, rank, src) for key, (rank, src) in best.items()]


def explode_keys(df, id_col, row_id_col, prefix="key"):
    records = []
    base_cols = [row_id_col, id_col]
    for _, row in df[base_cols].iterrows():
        for key, rank, source in id_candidates(row[id_col]):
            records.append(
                {
                    row_id_col: row[row_id_col],
                    f"{prefix}": key,
                    f"{prefix}_rank": rank,
                    f"{prefix}_source": source,
                }
            )
    return pd.DataFrame(records)


def build_stop_lookup(df_stops):
    if df_stops.empty:
        return pd.DataFrame()

    candidate_columns = [c for c in ["stop_id", "stop_code", "platform_code", "parent_station"] if c in df_stops.columns]
    if not candidate_columns:
        return pd.DataFrame()

    required = [c for c in ["typ", "stop_id", "stop_name", "stop_lat", "stop_lon"] if c in df_stops.columns]
    records = []

    for idx, row in df_stops.reset_index(drop=True).iterrows():
        for col_rank, col in enumerate(candidate_columns):
            for key, key_rank, source in id_candidates(row.get(col)):
                rec = {c: row.get(c) for c in required}
                rec.update(
                    {
                        "_stop_lookup_id": idx,
                        "stop_key": key,
                        "stop_key_rank": col_rank * 10 + key_rank,
                        "stop_key_source": f"{col}:{source}",
                    }
                )
                records.append(rec)

    lookup = pd.DataFrame(records)
    if lookup.empty:
        return lookup

    lookup = lookup.sort_values(["typ", "stop_key", "stop_key_rank"])
    lookup = lookup.drop_duplicates(subset=["typ", "stop_key"], keep="first")
    return lookup


def build_route_lookup(df_routes):
    if df_routes.empty:
        return pd.DataFrame()

    candidate_columns = [c for c in ["route_id", "route_short_name", "route_long_name"] if c in df_routes.columns]
    if not candidate_columns:
        return pd.DataFrame()

    records = []
    for idx, row in df_routes.reset_index(drop=True).iterrows():
        for col_rank, col in enumerate(candidate_columns):
            for key, key_rank, source in id_candidates(row.get(col)):
                records.append(
                    {
                        "_route_lookup_id": idx,
                        "typ": row.get("typ"),
                        "route_key": key,
                        "route_key_rank": col_rank * 10 + key_rank,
                        "route_key_source": f"{col}:{source}",
                        "route_id": row.get("route_id"),
                        "route_short_name": row.get("route_short_name", row.get("route_id")),
                    }
                )

    lookup = pd.DataFrame(records)
    if lookup.empty:
        return lookup

    lookup = lookup.sort_values(["typ", "route_key", "route_key_rank"])
    lookup = lookup.drop_duplicates(subset=["typ", "route_key"], keep="first")
    return lookup


def fetch_current_gtfs_static():
    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/GTFS_KRK_T.zip",
        "Autobusy": "https://gtfs.ztp.krakow.pl/GTFS_KRK_A.zip",
    }

    all_stops = []
    all_routes = []

    for v_type, url in urls.items():
        try:
            print(f"Pobieranie aktualnej bazy rozkładów GTFS dla: {v_type}...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                content = resp.read()

            with zipfile.ZipFile(io.BytesIO(content)) as z:
                with z.open("stops.txt") as f:
                    df_s = pd.read_csv(f, dtype=str)
                    df_s["typ"] = v_type
                    all_stops.append(df_s)

                with z.open("routes.txt") as f:
                    df_r = pd.read_csv(f, dtype=str)
                    df_r["typ"] = v_type
                    all_routes.append(df_r)

        except Exception as exc:
            print(f"Błąd pobierania statycznego GTFS dla {v_type}: {exc}")

    df_stops = pd.concat(all_stops, ignore_index=True) if all_stops else pd.DataFrame()
    df_routes = pd.concat(all_routes, ignore_index=True) if all_routes else pd.DataFrame()
    return df_stops, df_routes


def merge_delays_with_static(df_delays, df_stops, df_routes):
    if df_delays.empty:
        return pd.DataFrame(), pd.DataFrame()
    if df_stops.empty:
        print("Brak statycznego GTFS – nie da się przypiąć nazw i współrzędnych przystanków.")
        return pd.DataFrame(), pd.DataFrame()

    df_stops = df_stops.copy()
    for col in ["stop_lat", "stop_lon"]:
        if col in df_stops.columns:
            df_stops[col] = pd.to_numeric(df_stops[col], errors="coerce")

    stop_lookup = build_stop_lookup(df_stops)
    if stop_lookup.empty:
        print("Nie udało się zbudować lookupu przystanków ze statycznego GTFS.")
        return pd.DataFrame(), pd.DataFrame()

    d = df_delays.reset_index(drop=True).copy()
    d["_delay_row_id"] = range(len(d))
    delay_keys = explode_keys(d, "stop_id", "_delay_row_id", prefix="stop_key")

    merged = delay_keys.merge(stop_lookup, on="stop_key", how="inner", suffixes=("_delay", "_static"))
    merged = merged.merge(d, on="_delay_row_id", how="left", suffixes=("_static", ""))

    # Zostawiamy tylko dopasowania tego samego typu pojazdu i najlepszy wariant klucza dla każdego wiersza RT.
    merged = merged[merged["typ_static"] == merged["typ"]].copy()
    if merged.empty:
        print("Nie udało się połączyć TripUpdates z przystankami. Próbki kluczy do debugowania:")
        print("RT stop_id:", sorted(d["stop_id"].dropna().astype(str).unique()[:10]))
        print("GTFS stop_key:", sorted(stop_lookup["stop_key"].dropna().astype(str).unique()[:10]))
        return pd.DataFrame(), pd.DataFrame()

    merged["_match_rank"] = merged["stop_key_rank_delay"] + merged["stop_key_rank_static"]
    merged = merged.sort_values(["_delay_row_id", "_match_rank"])
    merged = merged.drop_duplicates(subset=["_delay_row_id"], keep="first")

    # Porządkujemy nazwy kolumn po merge'u.
    if "typ_static" in merged.columns:
        merged = merged.drop(columns=["typ_static"])
    if "stop_id_static" in merged.columns:
        merged = merged.rename(columns={"stop_id_static": "static_stop_id"})

    # Linie: też odporne łączenie, a gdy się nie uda – zostaje line_num z TripUpdates.
    merged["linia"] = merged["line_num"].astype(str)
    route_lookup = build_route_lookup(df_routes) if not df_routes.empty else pd.DataFrame()
    if not route_lookup.empty:
        route_base = merged.reset_index(drop=True).copy()
        route_base["_merged_row_id"] = range(len(route_base))
        route_keys = explode_keys(route_base.rename(columns={"line_num": "route_input"}), "route_input", "_merged_row_id", prefix="route_key")
        route_lookup_for_merge = route_lookup.rename(columns={"typ": "typ_route"})
        r = route_keys.merge(route_lookup_for_merge, on="route_key", how="left", suffixes=("_delay", "_static"))
        r = r.merge(route_base[["_merged_row_id", "typ"]], on="_merged_row_id", how="left")
        r = r[(r["typ_route"].isna()) | (r["typ_route"] == r["typ"])]
        r = r.sort_values(["_merged_row_id", "route_key_rank_delay", "route_key_rank_static"], na_position="last")
        r = r.drop_duplicates("_merged_row_id", keep="first")

        route_names = r.set_index("_merged_row_id")["route_short_name"].to_dict()
        merged = route_base.copy()
        merged["linia"] = merged["_merged_row_id"].map(route_names).fillna(merged["line_num"].astype(str))
        merged = merged.drop(columns=["_merged_row_id"])

    stop_cols = ["stop_name", "stop_lat", "stop_lon", "typ"]
    df_stops_delays = (
        merged.dropna(subset=["stop_name", "stop_lat", "stop_lon"])
        .groupby(stop_cols, as_index=False)
        .agg(
            mean_delay_min=("delay_min", "mean"),
            max_delay_min=("delay_min", "max"),
            measurements_count=("delay_min", "count"),
        )
    )

    return merged.reset_index(drop=True), df_stops_delays.reset_index(drop=True)


# Główne przygotowanie danych do mapy i wykresów.
df_stops = pd.DataFrame()
df_routes = pd.DataFrame()
df_merged = pd.DataFrame()
df_stops_delays = pd.DataFrame()

if not df_delays.empty:
    df_stops, df_routes = fetch_current_gtfs_static()
    df_merged, df_stops_delays = merge_delays_with_static(df_delays, df_stops, df_routes)

    print("\nPodsumowanie połączenia danych:")
    print(f"  obserwacje opóźnień: {len(df_delays)}")
    print(f"  obserwacje z nazwą i koordynatami przystanku: {len(df_merged)}")
    print(f"  unikalne przystanki w analizie: {len(df_stops_delays)}")

    if not df_stops_delays.empty:
        display(df_stops_delays.sort_values(by="mean_delay_min", ascending=False).head(15))
    elif not df_merged.empty:
        print("Dane połączone, ale brak agregacji po przystankach – sprawdź kolumny stop_name/stop_lat/stop_lon.")
    else:
        print("df_merged jest pusty – wykresy po liniach/godzinach użyją danych GTFS-RT bez koordynatów.")
else:
    print("Brak danych opóźnień – uruchom komórkę pobierania GTFS-RT ponownie.")


Pobieranie aktualnej bazy rozkładów GTFS dla: Tramwaje...
Pobieranie aktualnej bazy rozkładów GTFS dla: Autobusy...

Podsumowanie połączenia danych:
  obserwacje opóźnień: 298
  obserwacje z nazwą i koordynatami przystanku: 298
  unikalne przystanki w analizie: 270


,stop_name,stop_lat,stop_lon,typ,mean_delay_min,max_delay_min,measurements_count
239,Winnicka,50.01983,19.87218,Autobusy,75.0,75.0,1
30,Bolesława Śmiałego,50.01835,19.81735,Autobusy,75.0,75.0,1
199,Skawina Samborek Most,49.98884,19.81098,Autobusy,75.0,75.0,1
198,Skawina Rynek,49.97520,19.82791,Autobusy,75.0,75.0,1
201,Skawina Tyniecka,49.98431,19.81650,Autobusy,75.0,75.0,1
202,Skawina Tyniecka Osiedle,49.97802,19.82003,Autobusy,75.0,75.0,1
207,Skotniki Szkoła,50.01696,19.87609,Autobusy,75.0,75.0,1
206,Skawina Żwirownia,49.98241,19.80785,Autobusy,75.0,75.0,1
204,Skawina Wojska Polskiego,49.98683,19.81429,Autobusy,75.0,75.0,1
205,Skawina Żwirowa,49.98558,19.80445,Autobusy,75.0,75.0,1


## Interaktywna mapa ruchu komunikacji miejskiej
Mapa przedstawia opóźnienia komunikacji na danej drodze.


Nakładamy dane z przystanków na mapę i animujemy je z użyciem GeoJSON. Używamy OSMnx do znalezienia najbliższych dróg.

Kolory przedstawiają wielkość opóźnienia na drodze.

In [14]:
def _delay_to_color(delay, vmin=0, vmax=30):
    cmap = plt.get_cmap("RdYlGn_r")
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    return mcolors.to_hex(cmap(norm(max(vmin, min(float(delay), vmax)))))


def _static_stop_map(df_stops_plot):
    """Awaryjna mapa punktowa – działa nawet wtedy, gdy OSMnx/Overpass nie pobierze siatki ulic."""
    m = folium.Map(location=[50.0614, 19.9383], zoom_start=12, tiles="cartodbpositron")

    for _, row in df_stops_plot.iterrows():
        delay = float(row["mean_delay_min"])
        radius = max(5, min(20, 5 + delay / 2))
        folium.CircleMarker(
            location=[row["stop_lat"], row["stop_lon"]],
            radius=radius,
            color=_delay_to_color(delay),
            fill=True,
            fill_opacity=0.75,
            popup=(
                f"<b>{row['stop_name']}</b><br>"
                f"Typ: {row['typ']}<br>"
                f"Śr. opóźnienie: {delay:.1f} min<br>"
                f"Pomiarów: {int(row['measurements_count'])}"
            ),
        ).add_to(m)
    return m


if "df_merged" in locals() and not df_merged.empty:
    df_map = df_merged.dropna(subset=["stop_name", "stop_lat", "stop_lon", "hour"]).copy()
    df_map["stop_lat"] = pd.to_numeric(df_map["stop_lat"], errors="coerce")
    df_map["stop_lon"] = pd.to_numeric(df_map["stop_lon"], errors="coerce")
    df_map["delay_min"] = pd.to_numeric(df_map["delay_min"], errors="coerce")
    df_map = df_map.dropna(subset=["stop_lat", "stop_lon", "delay_min"])

    if df_map.empty:
        print("Brak poprawnych współrzędnych do mapy.")
    else:
        # Agregujemy punkt/godzinę, żeby mapa nie była przeładowana duplikatami.
        df_stop_hour = (
            df_map.groupby(["stop_name", "stop_lat", "stop_lon", "typ"], as_index=False)
            .agg(mean_delay_min=("delay_min", "mean"), measurements_count=("delay_min", "count"))

        )

        folium_map = folium.Map(location=[50.0614, 19.9383], zoom_start=13, tiles="cartodbdark_matter")
        try:
            lokalizacja = "Kraków, Poland"
            print(f"Pobieranie geometrii dróg dla: {lokalizacja}...")
            G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)

            unique_points = df_stop_hour.drop_duplicates(subset=["stop_name", "stop_lat", "stop_lon"]).reset_index(drop=True)
            unique_points["_point_id"] = range(len(unique_points))

            print("Przypinanie przystanków do najbliższych ulic...")
            try:
                nearest_edges = ox.nearest_edges(G, X=unique_points["stop_lon"].values, Y=unique_points["stop_lat"].values)
            except AttributeError:
                nearest_edges = ox.distance.nearest_edges(G, X=unique_points["stop_lon"].values, Y=unique_points["stop_lat"].values)

            point_to_edge = dict(zip(unique_points["_point_id"], nearest_edges))
            df_stop_hour = df_stop_hour.merge(
                unique_points[["_point_id", "stop_name", "stop_lat", "stop_lon"]],
                on=["stop_name", "stop_lat", "stop_lon"],
                how="left",
            )

            print("Budowanie animowanej warstwy GeoJSON...")
            for _, row in df_stop_hour.iterrows():
                if pd.isna(row["_point_id"]):
                    continue
                edge = point_to_edge.get(int(row["_point_id"]))
                if edge is None:
                    continue

                if len(edge) == 3:
                    u, v, key = edge
                    edge_data = G.get_edge_data(u, v, key) or {}
                else:
                    u, v = edge[:2]
                    data = G.get_edge_data(u, v) or {}
                    edge_data = next(iter(data.values())) if isinstance(data, dict) and data and "geometry" not in data else data

                if edge_data and "geometry" in edge_data:
                    coords = [[x, y] for x, y in edge_data["geometry"].coords]
                else:
                    coords = [[G.nodes[u]["x"], G.nodes[u]["y"]], [G.nodes[v]["x"], G.nodes[v]["y"]]]

                delay = float(row["mean_delay_min"])
                folium_coords = [[y, x] for x, y in coords]
                folium.PolyLine(
                    locations=folium_coords,
                    color=_delay_to_color(delay),
                    weight=7,
                    opacity=0.85,
                    tooltip=f"{row['stop_name']} – {delay:.1f} min"
                ).add_to(folium_map)

            if True:
                display(folium_map)
            else:
                print("Nie zbudowano warstw ulic – pokazuję mapę punktową przystanków.")
                display(_static_stop_map(df_stops_delays))

        except Exception as exc:
            print(f"OSMnx/Overpass nie zwrócił siatki ulic ({exc}). Pokazuję awaryjną mapę punktową.")
            display(_static_stop_map(df_stops_delays))
else:
    print("Brak danych z koordynatami (df_merged) do wyświetlenia mapy.")


Pobieranie geometrii dróg dla: Kraków, Poland...
Przypinanie przystanków do najbliższych ulic...
Budowanie animowanej warstwy GeoJSON...


## Wykresy (Gdzie są największe opóźnienia)
Przeanalizujmy, które przystanki oraz które linie notują średnio największe opóźnienia w pozyskanej próbce czasowej.

In [19]:
def show_bar(df, x, y, color, title, labels, height=500):
    if df.empty:
        print(f"Brak danych do wykresu: {title}")
        return None
    try:
        fig = px.bar(df, x=x, y=y, color=color, title=title, labels=labels, text_auto=".1f", height=height)
    except TypeError:
        fig = px.bar(df, x=x, y=y, color=color, title=title, labels=labels, height=height)
        fig.update_traces(texttemplate="%{y:.1f}", textposition="outside")
    fig.update_layout(xaxis_tickangle=-45)
    if x == "linia":
        fig.update_layout(xaxis_type="category")
    fig.show()
    return fig


# Top przystanków – bez filtra >=2, bo przy pojedynczym snapshotcie GTFS-RT często każdy przystanek ma tylko 1 pomiar.
if "df_stops_delays" in locals() and not df_stops_delays.empty:
    top_15_mean = df_stops_delays.sort_values(by="mean_delay_min", ascending=False).head(15)
    fig_bar_stops = show_bar(
        top_15_mean,
        x="stop_name",
        y="mean_delay_min",
        color="typ",
        title="Top 15 przystanków o największym średnim opóźnieniu",
        labels={"stop_name": "Przystanek", "mean_delay_min": "Średnie opóźnienie (min)", "typ": "Typ"},
    )
else:
    print("Brak danych przystankowych – wykres przystanków zostanie pominięty.")


# Top linii – używamy df_merged, a gdy brakuje statycznego GTFS, bierzemy same dane RT.
if "df_merged" in locals() and not df_merged.empty:
    route_source = df_merged.copy()
    if "linia" not in route_source.columns:
        route_source["linia"] = route_source["line_num"].astype(str)
elif "df_delays" in locals() and not df_delays.empty:
    route_source = df_delays.copy()
    route_source["linia"] = route_source["line_num"].astype(str)
else:
    route_source = pd.DataFrame()

if not route_source.empty:
    df_route_delays_all = route_source.groupby(["linia", "typ"], as_index=False).agg(
        mean_delay_min=("delay_min", "mean"),
        measurements_count=("delay_min", "count"),
        max_delay_min=("delay_min", "max"),
    )

    # Preferujemy linie z min. 3 pomiarami, ale nie robimy pustego wykresu – fallback bierze wszystkie.
    df_route_delays = df_route_delays_all[df_route_delays_all["measurements_count"] >= 3].copy()
    if df_route_delays.empty:
        df_route_delays = df_route_delays_all.copy()
        print("Za mało linii z ≥3 pomiarami – pokazuję ranking ze wszystkich dostępnych obserwacji.")

    top_15_routes = df_route_delays.sort_values(by="mean_delay_min", ascending=False).head(15)
    fig_bar_routes = show_bar(
        top_15_routes,
        x="linia",
        y="mean_delay_min",
        color="typ",
        title="Jakie linie najwięcej się spóźniają? (Top 15)",
        labels={"linia": "Numer linii", "mean_delay_min": "Średnie opóźnienie (min)", "typ": "Typ"},
    )
else:
    print("Brak danych do wykresu linii.")


# Opóźnienia po godzinach.
if not route_source.empty and "hour" in route_source.columns:
    df_hourly = route_source.groupby("hour", as_index=False).agg(
        avg_delay_min=("delay_min", "mean"),
        measurements_count=("delay_min", "count"),
    ).sort_values(by="hour")

    if len(df_hourly) == 1:
        fig_time = px.bar(
            df_hourly,
            x="hour",
            y="avg_delay_min",
            text="avg_delay_min",
            title="Średnie opóźnienie w aktualnej godzinie",
            labels={"hour": "Godzina", "avg_delay_min": "Średnie opóźnienie (min)"},
        )
        fig_time.update_traces(texttemplate="%{y:.1f}", textposition="outside")
    else:
        fig_time = px.line(
            df_hourly,
            x="hour",
            y="avg_delay_min",
            markers=True,
            title="W jakich godzinach jest najwięcej spóźnień KMK?",
            labels={"hour": "Godzina", "avg_delay_min": "Średnie opóźnienie (min)"},
        )
    fig_time.show()
else:
    print("Brak kolumny hour – wykres godzinowy zostanie pominięty.")


# Wykres rozkładu, żeby łatwo zobaczyć skalę danych z aktualnego snapshotu.
if not route_source.empty:
    fig_hist = px.histogram(
        route_source,
        x="delay_min",
        color="typ",
        nbins=30,
        title="Histogram opóźnień w aktualnej próbce",
        labels={"delay_min": "Opóźnienie (min)", "typ": "Typ"},
    )
    fig_hist.update_yaxes(title_text="Liczba obserwacji")
    fig_hist.show()
